# referral-audit: Kaggle quick start

Before running: **Settings > Accelerator > GPU T4 x2** and **Settings > Internet > On**.

Run the cells top to bottom. The first run on SCIN takes roughly 1 to 2 hours of GPU time (about 10k images x 50 variants). Start with `LIMIT = 500` to check everything works (about 5 minutes).

In [ ]:
%cd /kaggle/working
!rm -rf referral-audit
!git clone https://github.com/Kaitlyn-Tay/referral-audit.git
%cd referral-audit
!pip install -q -e '.[torch]'

In [ ]:
DATASET = 'scin'          # or 'fitzpatrick17k'
LIMIT = 500               # set to None for the full dataset
MODEL = 'vit_base_patch14_dinov2.lvd142m'
COVERAGE = 0.8
ALL_IMAGES = True         # SCIN: keep all photos per case (real-vs-synthetic check)
root = f'/kaggle/working/data/{DATASET}' + ('' if LIMIT is None else f'_{LIMIT}')
feat = f'/kaggle/working/features/{DATASET}_{LIMIT}.npz'
out = f'/kaggle/working/results/{DATASET}_{LIMIT}'
limit_arg = '' if LIMIT is None else f'--limit {LIMIT}'
all_arg = '--all-images' if (ALL_IMAGES and DATASET == 'scin') else ''

In [ ]:
!python scripts/download_data.py {DATASET} --root {root} {limit_arg} {all_arg}

In [ ]:
!python scripts/extract_features.py --manifest {root}/manifest.csv --out {feat} --model {MODEL} --batch-size 128 --workers 4

In [ ]:
!python scripts/run_audit.py --manifest {root}/manifest.csv --features {feat} --out {out} --coverage {COVERAGE}

In [ ]:
import pandas as pd
from IPython.display import Image, display

display(pd.read_csv(f'{out}/scores.csv').round(3))
display(Image(f'{out}/referral_by_group.png'))
display(Image(f'{out}/instability_by_factor.png'))

Features are saved after each variant, so if the session stops, re-run the extraction cell and it resumes. Save `/kaggle/working/features` as a Kaggle Dataset (Save Version > output) so later sessions can skip extraction.